# CELL 1

In [5]:
!pip install -q transformers torch accelerate pdfplumber pyngrok streamlit json-repair pandas sentencepiece bitsandbytes pytesseract pdf2image pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 57.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 99.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 92.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 74.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is 

In [ ]:
!apt-get update -y && apt-get install -y poppler-utils tesseract-ocr

In [1]:
# 1. Install Linux system packages for PDF rendering & Arabic/English OCR
!apt-get update -y && apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-ara

# 2. Install all required Python libraries with compatibility fixes
!pip install -q -U \
    "bitsandbytes>=0.46.1" \
    accelerate \
    transformers \
    "pillow==10.4.0" \
    pytesseract \
    pdf2image \
    pdfplumber \
    streamlit \
    pyngrok \
    json-repair \
    sentencepiece \
    pandas

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,845 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:11 https://cli.github.com/packages stable/main amd64 Packages [355 B]      
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1

# LOAD LLM 

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"

print("⏳ Downloading and caching 4-bit Mistral to local disk...")

# Download Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Download & Load Model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("\n✅ Model fully downloaded and cached on disk!")

⏳ Downloading and caching 4-bit Mistral to local disk...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]


✅ Model fully downloaded and cached on disk!


# CELL 2

In [ ]:
%%writefile app.py
import os
import re
import json
import sqlite3
from datetime import datetime
from PIL import Image

import streamlit as st
import pandas as pd
import pdfplumber
import pytesseract
from pdf2image import convert_from_bytes

DB_PATH = "invoices.db"

st.set_page_config(page_title="Al-Tazaj Invoice Ingestion", layout="wide")

# ---------------------------------------------------------
# TEXT EXTRACTION (English-only OCR - we never need the Arabic
# labels, and dropping "ara" from tesseract roughly halves OCR
# time and removes a lot of garbled noise from the raw text)
# ---------------------------------------------------------
ARABIC_RE = re.compile(r"[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]+")

def strip_arabic(text: str) -> str:
    return ARABIC_RE.sub("", text)

def extract_text_from_file(uploaded_file) -> str:
    file_type = uploaded_file.type

    if file_type in ["image/png", "image/jpeg", "image/jpg"]:
        image = Image.open(uploaded_file)
        return strip_arabic(pytesseract.image_to_string(image, lang="eng"))

    file_bytes = uploaded_file.read()
    uploaded_file.seek(0)

    text_chunks = []
    with pdfplumber.open(uploaded_file) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text_chunks.append(page_text.strip())

    combined_text = "\n".join(text_chunks).strip()
    combined_text = strip_arabic(combined_text)
    if len(combined_text) >= 50:
        return combined_text

    # scanned/image PDF -> OCR fallback, English only
    st.info("Scanned PDF detected. Running English-only OCR...")
    images = convert_from_bytes(file_bytes)
    ocr_chunks = []
    for i, img in enumerate(images):
        page_ocr = pytesseract.image_to_string(img, lang="eng")
        ocr_chunks.append(f"--- Page {i + 1} ---\n{page_ocr}")
    return strip_arabic("\n".join(ocr_chunks))


# ---------------------------------------------------------
# TEMPLATE-SPECIFIC DETERMINISTIC PARSER
# ---------------------------------------------------------
# All Al-Tazaj invoices share the exact same layout, so instead of
# sending the whole document (addresses, VAT/TRN numbers, QR code
# text, bilingual labels, etc.) to an LLM and waiting on it, we pull
# out only what's needed with regex. This is instant and has zero
# hallucination risk. No GPU / model load required for the normal case.

INVOICE_NO_RE = re.compile(r"Invoice\s*No\s*:?\s*([0-9]+)", re.IGNORECASE)
INVOICE_DATE_RE = re.compile(r"Invoice\s*Issue\s*Date\s*:?\s*([0-9]{4}-[0-9]{2}-[0-9]{2})", re.IGNORECASE)

UNIT_ALTERNATION = r"PCS|KG|PKT|BOX"
# Numeric fields are deliberately tolerant of a missing decimal point
# (e.g. OCR sometimes reads "7.15" as "215") so a single misread digit
# doesn't drop the whole row - we flag those rows as low_confidence
# instead, via a cross-check against qty * unit_price.
ITEM_LINE_RE = re.compile(
    r"^\s*(?P<line_no>\d{1,3})\s+"
    r"(?P<description>.+?)\s+"
    r"(?P<unit>" + UNIT_ALTERNATION + r")\s+"
    r"(?P<qty>[\d,]+(?:\.\d+)?)\s+"
    r"(?P<unit_price>[\d,]+(?:\.\d+)?)\s+"
    r"(?P<amount>[\d,]+(?:\.\d+)?)\s+"
    r"(?P<vat_rate>[\d.]+)\s+"
    r"(?P<vat_amount>[\d,]+(?:\.\d+)?)\s+"
    r"(?P<total>[\d,]+(?:\.\d+)?)[\}\]\)]?\s*$",
    re.MULTILINE,
)


def _to_float(s: str) -> float:
    try:
        return float(s.replace(",", "").rstrip("}])"))
    except (ValueError, AttributeError):
        return 0.0


# ---------------------------------------------------------
# CATEGORIZATION - rule-based, instant, no model call
# ---------------------------------------------------------
# The Al-Tazaj catalog is a fixed, recurring set of products, so instead
# of asking an LLM to categorize every row on every invoice, we match on
# keywords once. This is instant and 100% consistent invoice-to-invoice.
# Order matters: more specific buckets are checked before broader ones
# (e.g. "coating" before "chicken", since "Broast Coating" contains
# "broast" but isn't actually a chicken item).
CATEGORY_RULES = [
    ("Coating & Ingredients", ["coating"]),
    ("Beverages", ["juice"]),
    ("Desserts", ["basbousa", "dessert"]),
    ("Burgers & Sandwiches", ["burger"]),
    ("Salads & Vegetables", ["salad", "lettuce", "tomato", "coleslaw"]),
    ("Sauces & Condiments", [
        "sauce", "dressing", "tahina", "mustard", "garlic", "dynamite",
        "tikka", "cocktail", "barbecue", "marination", "homos", "hummus",
        "muhammas",
    ]),
    ("Chicken", ["chicken", "broast", "farooj", "tandoori"]),
]


def get_category(description: str) -> str:
    d = description.lower()
    for category, keywords in CATEGORY_RULES:
        if any(kw in d for kw in keywords):
            return category
    return "Uncategorized"


def llm_categorize(descriptions: list) -> dict:
    """Only called for the rare item that doesn't match any keyword rule.
    Small prompt, tiny output - one short category label per description."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from json_repair import repair_json

    MODEL_NAME = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"

    @st.cache_resource(show_spinner=False)
    def _load():
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config, device_map="auto", low_cpu_mem_usage=True
        )
        return tok, mdl

    tokenizer, model = _load()
    existing_cats = ", ".join(c for c, _ in CATEGORY_RULES)
    prompt = (
        f"Assign each of these food-supplier invoice line item descriptions to a short "
        f"category. Prefer one of these existing categories if it fits: {existing_cats}. "
        f"Only invent a new short category name if none of those fit.\n"
        f"Return ONLY a JSON object mapping description -> category, nothing else.\n\n"
        + json.dumps(descriptions)
    )
    formatted = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=60 * len(descriptions) + 100,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    cleaned = re.sub(r"```json|```", "", decoded).strip()
    try:
        return json.loads(repair_json(cleaned))
    except Exception:
        return {}


def parse_tazaj_invoice(text: str) -> dict:
    """Deterministic parse of the fixed Al-Tazaj invoice template.
    Returns invoice_number, invoice_date, and every line item field -
    only the surrounding page noise (addresses, TRNs, QR code, logos,
    bilingual labels) is ignored."""
    inv_match = INVOICE_NO_RE.search(text)
    date_match = INVOICE_DATE_RE.search(text)

    invoice_number = inv_match.group(1) if inv_match else None
    invoice_date = date_match.group(1) if date_match else None

    line_items = []
    for m in ITEM_LINE_RE.finditer(text):
        qty = _to_float(m.group("qty"))
        unit_price = _to_float(m.group("unit_price"))
        amount = _to_float(m.group("amount"))
        # sanity check: qty * unit_price should roughly equal the stated
        # amount; if it's way off, a digit was probably misread by OCR
        expected = qty * unit_price
        low_confidence = amount > 0 and abs(expected - amount) / amount > 0.15
        description = m.group("description").strip()

        line_items.append({
            "line_no": int(m.group("line_no")),
            "description": description,
            "category": get_category(description),
            "unit": m.group("unit"),
            "qty": qty,
            "unit_price": unit_price,
            "amount_without_vat": amount,
            "vat_rate": _to_float(m.group("vat_rate")),
            "vat_amount": _to_float(m.group("vat_amount")),
            "total_amount": _to_float(m.group("total")),
            "low_confidence": low_confidence,
        })

    # rare fallback: only for items no keyword rule matched
    uncategorized = sorted({it["description"] for it in line_items if it["category"] == "Uncategorized"})
    if uncategorized:
        try:
            mapping = llm_categorize(uncategorized)
            for it in line_items:
                if it["description"] in mapping:
                    it["category"] = mapping[it["description"]]
        except Exception:
            pass  # keep "Uncategorized" rather than blocking the whole invoice

    return {
        "invoice_number": invoice_number,
        "invoice_date": invoice_date,
        "line_items": line_items,
    }


# ---------------------------------------------------------
# OPTIONAL LLM FALLBACK
# ---------------------------------------------------------
# Only used if the regex parser finds zero line items - e.g. someone
# uploads a document that isn't this Al-Tazaj template. Kept lazy so
# torch/transformers are never loaded (no GPU wait) on the common path.
def llm_fallback_extract(ocr_text: str) -> dict:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from json_repair import repair_json

    MODEL_NAME = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"

    @st.cache_resource(show_spinner=False)
    def _load():
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config, device_map="auto", low_cpu_mem_usage=True
        )
        return tok, mdl

    tokenizer, model = _load()

    prompt = (
        "Extract invoice_number, invoice_date, and every line item "
        "(line_no, description, category, unit, qty, unit_price, amount_without_vat, "
        "vat_rate, vat_amount, total_amount) from this OCR text as a single "
        "JSON object with keys invoice_number, invoice_date, line_items. "
        "Deduce 3 to 7 short, logical categories for the items and assign each item one. "
        "Return ONLY the JSON.\n\n" + ocr_text[:8000]
    )
    formatted = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=4096, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    cleaned = re.sub(r"```json|```", "", decoded).strip()
    return json.loads(repair_json(cleaned))


# ---------------------------------------------------------
# DATABASE LAYER (only what we actually keep: id, date, items)
# ---------------------------------------------------------
def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def init_db():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS invoices (
            invoice_number TEXT PRIMARY KEY,
            invoice_date TEXT
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS line_items (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            invoice_number TEXT,
            line_no INTEGER,
            description TEXT,
            category TEXT,
            unit TEXT,
            qty REAL,
            unit_price REAL,
            amount_without_vat REAL,
            vat_rate REAL,
            vat_amount REAL,
            total_amount REAL,
            low_confidence INTEGER DEFAULT 0,
            FOREIGN KEY (invoice_number) REFERENCES invoices (invoice_number)
        )
    """)
    conn.commit()
    conn.close()


def save_invoice(data: dict):
    conn = get_conn()
    cur = conn.cursor()

    raw_inv_num = str(data.get("invoice_number") or "").strip()
    if not raw_inv_num or raw_inv_num.lower() in ["null", "none", "n/a", "unknown", "0"]:
        invoice_number = f"INV-AUTO-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
        data["invoice_number"] = invoice_number
    else:
        invoice_number = raw_inv_num

    cur.execute("""
        INSERT INTO invoices (invoice_number, invoice_date)
        VALUES (?, ?)
        ON CONFLICT(invoice_number) DO UPDATE SET
            invoice_date=excluded.invoice_date
    """, (invoice_number, data.get("invoice_date", "")))

    cur.execute("DELETE FROM line_items WHERE invoice_number = ?", (invoice_number,))

    for item in data.get("line_items", []):
        cur.execute("""
            INSERT INTO line_items (
                invoice_number, line_no, description, category, unit,
                qty, unit_price, amount_without_vat, vat_rate, vat_amount, total_amount,
                low_confidence
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            invoice_number,
            item.get("line_no"),
            item.get("description", ""),
            item.get("category", "Uncategorized"),
            item.get("unit"),
            float(item.get("qty") or 0.0),
            float(item.get("unit_price") or 0.0),
            float(item.get("amount_without_vat") or 0.0),
            float(item.get("vat_rate") or 0.0),
            float(item.get("vat_amount") or 0.0),
            float(item.get("total_amount") or 0.0),
            int(bool(item.get("low_confidence"))),
        ))

    conn.commit()
    conn.close()


def fetch_invoice_options():
    conn = get_conn()
    df = pd.read_sql_query("SELECT invoice_number, invoice_date FROM invoices ORDER BY invoice_date, invoice_number", conn)
    conn.close()
    return df

def fetch_line_items_for_invoice(invoice_number: str) -> pd.DataFrame:
    conn = get_conn()
    df = pd.read_sql_query("SELECT * FROM line_items WHERE invoice_number = ? ORDER BY line_no", conn, params=(invoice_number,))
    conn.close()
    return df

def fetch_all_line_items() -> pd.DataFrame:
    conn = get_conn()
    df = pd.read_sql_query("SELECT * FROM line_items", conn)
    conn.close()
    return df

def aggregate_across_invoices(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    agg = df.groupby("description", as_index=False).agg(
        category=("category", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
        unit=("unit", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
        qty=("qty", "sum"),
        amount_without_vat=("amount_without_vat", "sum"),
        vat_amount=("vat_amount", "sum"),
        total_amount=("total_amount", "sum"),
    )
    agg["unit_price"] = agg.apply(lambda r: round(r["amount_without_vat"] / r["qty"], 4) if r["qty"] else 0.0, axis=1)
    cols = ["description", "category", "unit", "qty", "unit_price", "amount_without_vat", "vat_amount", "total_amount"]
    return agg[cols].sort_values(["category", "description"]).reset_index(drop=True)


def render_category_tables(df: pd.DataFrame):
    if df.empty:
        st.info("No line items to display.")
        return
    categories = sorted(df["category"].dropna().unique().tolist())
    for cat in categories:
        st.subheader(f"\U0001F4E6 {cat}")
        subset = df[df["category"] == cat].reset_index(drop=True)
        st.dataframe(subset, use_container_width=True)


# ---------------------------------------------------------
# UI
# ---------------------------------------------------------
init_db()
st.title("Al-Tazaj Invoice Ingestion (fast, template-specific parser)")

tab1, tab2 = st.tabs(["Upload & Ingest", "Database Viewer & Aggregator"])

with tab1:
    st.subheader("Upload an invoice (PDF or Image)")
    uploaded_file = st.file_uploader("Choose a PDF, PNG, or JPG file", type=["pdf", "png", "jpg", "jpeg"])

    if uploaded_file is not None:
        # Only re-run OCR/text extraction when a genuinely NEW file is
        # uploaded. Without this, Streamlit's full-script rerun on every
        # widget interaction (e.g. editing a cell in the review table
        # below) would re-extract text from the same file every time.
        file_key = f"{uploaded_file.name}_{uploaded_file.size}"
        if st.session_state.get("ocr_file_key") != file_key:
            with st.spinner("Extracting text..."):
                st.session_state["ocr_text"] = extract_text_from_file(uploaded_file)
            st.session_state["ocr_file_key"] = file_key
        ocr_text = st.session_state["ocr_text"]

        with st.expander("Extracted text preview (Arabic stripped)", expanded=False):
            st.text(ocr_text[:5000] + ("..." if len(ocr_text) > 5000 else ""))

        if st.button("Process Invoice", type="primary"):
            data = parse_tazaj_invoice(ocr_text)

            if not data["line_items"]:
                st.warning("Template parser found no line items - falling back to the LLM. This is slower.")
                with st.spinner("Running LLM fallback extraction..."):
                    try:
                        data = llm_fallback_extract(ocr_text)
                    except Exception as e:
                        st.error(f"LLM fallback also failed: {e}")
                        st.stop()
            else:
                n_items = len(data["line_items"])
                n_low_conf = sum(1 for it in data["line_items"] if it.get("low_confidence"))
                st.success(f"Parsed instantly: invoice {data['invoice_number']}, {n_items} line items.")
                if n_low_conf:
                    st.warning(f"{n_low_conf} row(s) flagged low_confidence - "
                               f"qty x unit_price didn't match the stated amount, "
                               f"usually an OCR misread digit. Worth a quick manual check.")

            # Stash the freshly-parsed invoice in session state instead of
            # saving right away. It survives the reruns triggered by editing
            # the review table below, so edits aren't lost and the invoice
            # doesn't get re-parsed on every widget interaction.
            st.session_state["review_invoice_number"] = data.get("invoice_number") or ""
            st.session_state["review_invoice_date"] = data.get("invoice_date") or ""
            st.session_state["review_items"] = pd.DataFrame(data["line_items"])

    # ---------------------------------------------------------
    # REVIEW & CORRECT step
    # ---------------------------------------------------------
    # Nothing reaches the database until "Save to Database" below is
    # clicked - this replaces the old auto-save-on-parse behavior.
    if "review_items" in st.session_state:
        st.divider()
        st.subheader("Review & correct before saving")
        st.caption(
            "Fix any OCR mistakes, add missing rows, or delete rows that "
            "shouldn't be here. Nothing is saved to the database until you "
            "click \"Save to Database\"."
        )

        header_col1, header_col2 = st.columns(2)
        with header_col1:
            edited_invoice_number = st.text_input(
                "Invoice number", value=st.session_state["review_invoice_number"]
            )
        with header_col2:
            edited_invoice_date = st.text_input(
                "Invoice date (YYYY-MM-DD)", value=st.session_state["review_invoice_date"]
            )

        st.caption("Double-click a cell to edit it. Use the + row at the bottom to add an "
                   "item, or the trash icon on a row to remove it.")
        edited_items_df = st.data_editor(
            st.session_state["review_items"],
            num_rows="dynamic",  # lets rows be added or deleted, not just edited
            use_container_width=True,
            key="line_items_editor",
        )

        save_col, discard_col = st.columns(2)
        with save_col:
            if st.button("Save to Database", type="primary"):
                final_data = {
                    "invoice_number": edited_invoice_number.strip(),
                    "invoice_date": edited_invoice_date.strip(),
                    "line_items": edited_items_df.to_dict("records"),
                }
                try:
                    save_invoice(final_data)
                    st.success("Invoice saved to database successfully!")
                    del st.session_state["review_items"]
                    del st.session_state["review_invoice_number"]
                    del st.session_state["review_invoice_date"]
                    st.rerun()
                except Exception as e:
                    st.error(f"Database insertion error: {e}")
        with discard_col:
            if st.button("Discard"):
                del st.session_state["review_items"]
                del st.session_state["review_invoice_number"]
                del st.session_state["review_invoice_date"]
                st.rerun()

with tab2:
    st.subheader("Database Viewer")

    if st.button("Refresh Database"):
        st.rerun()

    options_df = fetch_invoice_options()

    if options_df.empty:
        st.info("No invoices in the database yet. Upload and process one in Tab 1 first.")
    else:
        choice_labels = ["All Invoices"] + [
            f"Invoice {row.invoice_number} - {row.invoice_date}" for row in options_df.itertuples()
        ]
        selected = st.selectbox("Select invoice view", choice_labels)

        if selected == "All Invoices":
            st.markdown("### Aggregated view across ALL invoices (by category)")
            all_items = fetch_all_line_items()
            aggregated = aggregate_across_invoices(all_items)
            render_category_tables(aggregated)

            st.markdown("---")
            st.markdown("### All invoices summary")
            conn = get_conn()
            inv_summary = pd.read_sql_query("SELECT * FROM invoices", conn)
            conn.close()
            st.dataframe(inv_summary, use_container_width=True)
        else:
            invoice_number = selected.split(" - ")[0].replace("Invoice ", "").strip()
            st.markdown(f"### Line items for Invoice {invoice_number}")
            items = fetch_line_items_for_invoice(invoice_number)
            render_category_tables(items)


# CELL 3

In [ ]:
# =========================================================
# CELL 3 — TUNNELING & EXECUTION
# =========================================================

import os
import time
import subprocess
import getpass
from pyngrok import ngrok

# 1. Authenticate ngrok
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN") or getpass.getpass("Enter your ngrok auth token: ")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 2. Terminate old active sessions
ngrok.kill()

# 3. Start Streamlit server in background
with open("streamlit.log", "w") as log_file:
    process = subprocess.Popen(
        [
            "streamlit", "run", "app.py",
            "--server.port", "8501",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false"
        ],
        stdout=log_file,
        stderr=log_file
    )

print("⏳ Starting Streamlit server...")
time.sleep(8)

# 4. Expose public ngrok link
public_url = ngrok.connect(8501, "http")
print("=" * 65)
print(f"🚀 Streamlit app is live at: {public_url}")
print("=" * 65)